### Deep learning model training.    

#### Top accuracy(full patch size：448x448):
|model|miou|oa|update_time|    
|----|----|----|----|   
|Swin-UNet|0.873|0.934|20260428|    
|Swin-U2Net|0.904|0.944|20260502|    
||%|%||    
||%|%||    
|||||  

#### Top accuracy(full patch size：512x512):
|model|miou|oa|update_time|note|
|----|----|----|----|----|   
|Swin-UNet|0.894|0.940|20260502||    
|Trans-UNet|0.901|0.946|20260502||    
|u2net-swin-fusion|0.946|0.989|20260525||    
|u2net-swin-fusion+cbam|0.946|0.989|20260525||    
||||||    
||||||  

### u3net 
|model|miou|oa|update_time|note|
|----|----|----|----|----|   
|u3net-swin-fusion|0.950|0.989|20260618||    
||||||    

### conclusions:   



In [1]:
import time
import torch
import random
import json
import pandas as pd
import torch.nn as nn
from glob import glob
from notebooks import config
import torch.nn.functional as F
import matplotlib.pyplot as plt
from utils.imgShow import imsShow
from torchvision.transforms import v2
from utils.data_aug import GaussianNoise
from utils.dataloader import read_scenes 
from utils.dataloader import SceneArraySet, PatchPathSet
from model import unet, swin_unet, u3net_swin_fusion, u3net_swin_fusion_, transunet
from torchmetrics.classification import BinaryJaccardIndex, BinaryAccuracy



In [2]:
patch_size = 512  ## patch size setting
batch_size_tra = 4  ## batch size setting
batch_size_val = 16  ## batch size setting
patch_resize = None  ## patch resize setting
### traset
paths_scene_tra, paths_truth_tra = config.paths_scene_tra, config.paths_truth_tra
paths_dem_tra = config.paths_dem_tra
print(f'train scenes: {len(paths_scene_tra)}')
## valset
paths_valset = sorted(glob(f'data/dset/valset/patch_{patch_size}/*'))  ## for model prediction 
print(f'vali patch {patch_size}: {len(paths_valset)}')
path_valset_lat = f'data/dset/valset/patch_{patch_size}/patches_lat.json'  ## for model prediction


train scenes: 52
vali patch 512: 118


### dataset loading

In [3]:
scenes_arr, truths_arr, scenes_lat = read_scenes(paths_scene_tra, paths_truth_tra, paths_dem_tra, lat=True)  
patches_lat = json.load(open(path_valset_lat, 'r'))  ## for model prediction


In [4]:
# discrete_rotation = v2.RandomChoice([
#     v2.RandomRotation(degrees=[0, 0]),
#     v2.RandomRotation(degrees=[90, 90]),
#     v2.RandomRotation(degrees=[180, 180]),
#     v2.RandomRotation(degrees=[270, 270]),
# ])
transforms_tra = v2.Compose([   
            v2.ToImage(),   
            v2.RandomCrop(size=(patch_size, patch_size)),   
            v2.RandomHorizontalFlip(p=0.3),   
            v2.RandomVerticalFlip(p=0.3),   
            v2.RandomApply([v2.RandomRotation(degrees=15)], p=0.3),   # type: ignore
            # discrete_rotation,  
            GaussianNoise(mean = 0.0, sigma_max_img=0.1, sigma_max_dem=0, p=0.3)
            ]) 
transforms_val = v2.Compose([v2.ToDtype(torch.float32)])   


In [5]:
# Create dataset instances
tra_data = SceneArraySet(scenes_arr=scenes_arr, truths_arr=truths_arr, 
                          patch_size=patch_size, transforms=transforms_tra, scenes_lat=scenes_lat)
val_data = PatchPathSet(paths_valset=paths_valset, transforms=transforms_val, path_valset_lat=path_valset_lat)

## Create data loaders
tra_loader = torch.utils.data.DataLoader(tra_data, batch_size=batch_size_tra, 
                                         shuffle=True, num_workers=5)
val_loader = torch.utils.data.DataLoader(val_data, batch_size=batch_size_val, num_workers=5) 



#### Model training

In [6]:
### check model
# model = swin_unet(img_size=512, num_bands=7, window_size=8)
# model = transunet(img_size=512, num_bands=7)
# model = swin_unet_2(img_size=512, num_bands=7, window_size=8)
# model = u2net_swin_fusion(num_bands_b1=6, num_bands_b2=1, 
#                               image_size=patch_size, window_size=8)
model = u3net_swin_fusion_(# backbone_name='resnet50', 
                            backbone_name='efficientnet_b0',
                            pretrained=True) 


Unexpected keys (bn2.num_batches_tracked, bn2.bias, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.
Unexpected keys (bn2.num_batches_tracked, bn2.bias, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.
Unexpected keys (bn2.num_batches_tracked, bn2.bias, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


Output channels from encoder stages: [16, 24, 40, 112, 320]


In [8]:
input_tensor = torch.randn(2, 7, 512, 512) 
output = model(input_tensor)  
print(output.shape)  


torch.Size([2, 1, 512, 512])


In [9]:
### create loss and optimizer
# loss_bce = nn.BCELoss()
creterion = nn.BCEWithLogitsLoss()
# optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)  ## for pretrained model
# lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, \
#                                       mode='min', 
#                                       factor=0.5, 
#                                       patience=20,
#                                       min_lr=1e-5)


In [10]:
'''------train loops------'''
def train_loops(model, loss_fn, 
                    optimizer, 
                    tra_loader, 
                    val_loader,                     
                    epoches, 
                    device, 
                    lr_scheduler=None):
    loss_tra_loops, miou_tra_loops, oa_tra_loops = [], [], []
    loss_val_loops, miou_val_loops, oa_val_loops = [], [], []
    model = model.to(device)
    size_tra_loader = len(tra_loader)
    size_val_loader = len(val_loader)
    for epoch in range(epoches):
        start = time.time()
        loss_tra, loss_val = 0, 0
        '''-----train the model-----'''
        miou_tra = BinaryJaccardIndex().to(device)
        oa_tra = BinaryAccuracy().to(device)
        model.train()   # training mode for dropout and batchnorm
        for x_batch, y_batch, lat_batch in tra_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            pred = model(x_batch)
            loss = loss_fn(pred, y_batch)
            loss.backward()
            optimizer.step()
            pred = (F.sigmoid(pred) > 0.5).float()
            miou_tra.update(pred, y_batch.long())
            oa_tra.update(pred, y_batch.long())
            loss_tra += loss.item()
        miou_tra_global = miou_tra.compute()
        oa_tra_global = oa_tra.compute()
        loss_tra_global = loss_tra/size_tra_loader
        miou_tra.reset(); oa_tra.reset()

        '''----- validation the model: time consuming -----'''
        oa_val = BinaryAccuracy().to(device)
        miou_val = BinaryJaccardIndex().to(device)
        model.eval()
        if epoch>500 and (epoch+1) % 3 == 0: 
            for x_batch, y_batch, lat_batch in val_loader:
                x_batch, y_batch, lat_batch = x_batch.to(device), y_batch.to(device), lat_batch.to(device)
                with torch.no_grad():
                    pred = model(x_batch)
                    # if x.shape[2] > 256:  ### crop inner 256x256 for evaluation
                    #     pred = v2.functional.center_crop(pred, 256)
                    #     y = v2.functional.center_crop(y, 256)
                    loss = loss_fn(pred, y_batch)
                pred = (F.sigmoid(pred) > 0.5).float()
                miou_val.update(pred, y_batch.long())
                oa_val.update(pred, y_batch.long())
                loss_val += loss.item()
            miou_val_global = miou_val.compute()
            oa_val_global = oa_val.compute()
            loss_val_global = loss_val/size_val_loader
            miou_val.reset(); oa_val.reset()

            loss_tra_loops.append(loss_tra_global); miou_tra_loops.append(miou_tra_global); oa_tra_loops.append(oa_tra_global)
            loss_val_loops.append(loss_val_global); miou_val_loops.append(miou_val_global); oa_val_loops.append(oa_val_global)
            print(f'Ep{epoch}: tra-> Loss:{loss_tra_global:.3f},Oa:{oa_tra_global:.3f},Miou:{miou_tra_global:.3f}, '
                    f'val-> Loss:{loss_val_global:.3f},Oa:{oa_val_global:.3f}, Miou:{miou_val_global:.3f},time:{time.time()-start:.1f}s')
        else: 
            print(f'Ep{epoch}: tra->Loss:{loss_tra_global:.3f},Oa:{oa_tra_global:.3f},Miou:{miou_tra_global:.3f}, \
                                time:{time.time()-start:.1f}s')
        if lr_scheduler:
          lr_scheduler.step(miou_tra_global)    ## if use lr_scheduler like ReduceLROnPlateau
        ## show the result
        if (epoch+1)%200 == 0:
            sam_index = random.randrange(len(val_data))
            patch, truth, _ = val_data[sam_index]
            patch, truth = patch.unsqueeze(0).to(device), truth.to(device)
            pred = model(patch)
            pred = F.sigmoid(pred)  ## convert logit to prob for metric calculation
            if patch.shape[2] > 256:  ## zoom in for visualization if patch size > 256
                pred_val = v2.functional.center_crop(pred, 256)
                patch_val = v2.functional.center_crop(patch, 256)
                truth_val = v2.functional.center_crop(truth, 256)
            else:
                patch_val = patch
                pred_val = pred
                truth_val = truth
            ## convert to numpy and plot
            patch = patch[0].to('cpu').detach().numpy().transpose(1,2,0)            
            pred = pred[0].to('cpu').detach().numpy()
            patch_val = patch_val[0].to('cpu').detach().numpy().transpose(1,2,0)
            pred_val = pred_val[0].to('cpu').detach().numpy()
            truth_val = truth_val.to('cpu').detach().numpy()
            imsShow([patch, pred, patch_val, pred_val, truth_val], 
                    clip_list = (2,0,2,0,0),
                    img_name_list=['input_patch', 'pred', 'patch_zoom_in', 'pred_zoom_in', 'truth_zoom_in'], 
                    figsize=(15, 3))
            plt.tight_layout() 
    metrics = {'tra_loss':loss_tra_loops, 'tra_oa': oa_tra_loops, 'tra_miou': miou_tra_loops,
                'val_loss': loss_val_loops, 'val_oa': oa_val_loops, 'val_miou': miou_val_loops}
    return metrics 


In [ ]:
device = torch.device('cuda:0')  
metrics = train_loops(model=model,  
                epoches=1000,  
                loss_fn=creterion,  
                optimizer=optimizer,     
                tra_loader=tra_loader,  
                val_loader=val_loader,  
                # lr_scheduler=lr_scheduler,  
                device=device)  


Ep0: tra->Loss:0.614,Oa:0.732,Miou:0.446,                                 time:3.3s
Ep1: tra->Loss:0.476,Oa:0.931,Miou:0.783,                                 time:2.4s
Ep2: tra->Loss:0.409,Oa:0.938,Miou:0.747,                                 time:2.4s


In [ ]:
# # model saving 
# model_name = 'u2net' 
# # model_name = 'u2net_biatt' 
# # model_name = 'u2net_cbam' 
# # model_name = 'deeplabv3plus'  
# # model_name = 'deeplabv3plus_mb2'  
# date_str = time.strftime("%Y-%m-%d-%H", time.localtime())  
# date_str = date_str.replace('-', '')  ## remove '-' for file name  
# path_save = f'model/trained/{model_name}/{model_name}_eval_mode.pth'   
# # path_save = f'model/trained/seg_models/{model_name}_weights_{date_str}.pth'   
# torch.save(model.state_dict(), path_save)     ## save weights of the trained model 
# ## model.load_state_dict(torch.load(path_save, weights_only=True))  ## load the weights of the trained model
# ## metrics saving
# path_metrics = f'model/trained/{model_name}/{model_name}_eval_mode_metrics.csv'    
# ## path_metrics = f'model/trained/seg_models/{model_name}_metrics_{date_str}.csv'    
# metrics_df = pd.DataFrame(metrics)
# metrics_df.to_csv(path_metrics, index=False, sep=',')
